In [ ]:
# Example 1: verify the public API surface declared by __all__.
import ray_log_notebook

exported = getattr(ray_log_notebook, '__all__', None)
print(f"__all__ = {exported}")

assert exported == ['enable'], f"expected __all__ == ['enable'], got {exported!r}"
assert hasattr(ray_log_notebook, 'enable'), "missing top-level enable()"
assert callable(ray_log_notebook.enable), "enable is not callable"
print("public API surface OK")

In [ ]:
# Example 2: enable() is re-exported from the capture submodule.
import ray_log_notebook
from ray_log_notebook import capture
from ray_log_notebook.capture import enable as enable_from_sub

assert ray_log_notebook.enable is enable_from_sub, "enable should be re-exported from capture"
assert hasattr(capture, 'RayLogCapture'), "capture.RayLogCapture missing"
assert hasattr(capture, 'logging_worker'), "capture.logging_worker missing"
print(f"capture submodule exposes: enable, RayLogCapture, logging_worker")

In [ ]:
# Example 3: introspect enable() signature + docstring.
import inspect

from ray_log_notebook import enable

sig = inspect.signature(enable)
doc = inspect.getdoc(enable) or ''

print(f"enable signature: {sig}")
print(f"enable doc: {doc}")

# enable() takes no arguments.
assert len(sig.parameters) == 0, f"expected no parameters, got {sig.parameters}"
assert 'jupyter' in doc.lower() or 'notebook' in doc.lower(), "doc should mention notebook/jupyter"
print("signature OK")

In [ ]:
# Example 4: outside a running event loop, enable() prints a message and returns instead of raising.
import subprocess
import sys

script = (
    "import ray_log_notebook;"
    "ray_log_notebook.enable();"
    "print('returned cleanly')"
)
result = subprocess.run(
    [sys.executable, '-c', script], capture_output=True, text=True, timeout=60
)
print('stdout:', result.stdout.strip())
print('stderr:', result.stderr.strip()[:300])

assert result.returncode == 0, f"enable() raised: {result.stderr}"
assert 'returned cleanly' in result.stdout, "enable() did not return"
print("enable() handles missing-loop case without raising")

In [ ]:
# Example 5: RayLogCapture instantiates against a fresh asyncio loop without touching Ray.
import asyncio

from ray_log_notebook.capture import RayLogCapture

loop = asyncio.new_event_loop()
try:
    capture_instance = RayLogCapture(loop)
    # The constructor wires up two asyncio.Queue objects and stashes the loop.
    assert capture_instance._loop is loop, "loop was not stored"
    assert hasattr(capture_instance, '_q_stdout'), "_q_stdout missing"
    assert hasattr(capture_instance, '_q_stderr'), "_q_stderr missing"
    # The capture object exposes its stdout/stderr and IPython/Ray hooks.
    for attr in ('put_stdout', 'put_stderr', 'pre_run_cell', 'register_ray', 'register_ipython'):
        assert callable(getattr(capture_instance, attr)), f"{attr} missing"
    print('RayLogCapture instantiation + API surface OK')
finally:
    loop.close()